LABELS: \
Cattle_ID, Breed, Climate_Zone, Management_System, Age_Months, Weight_kg,
Parity, Lactation_Stage, Days_in_Milk, Feed_Type, Feed_Quantity_kg,
Feeding_Frequency, Water_Intake_L, Walking_Distance_km, Grazing_Duration_hrs,
Rumination_Time_hrs, Resting_Hours, Ambient_Temperature_C, Humidity_percent,
Housing_Score, FMD_Vaccine, Brucellosis_Vaccine, HS_Vaccine, BQ_Vaccine,
Anthrax_Vaccine, IBR_Vaccine, BVD_Vaccine, Rabies_Vaccine,
Previous_Week_Avg_Yield, Body_Condition_Score, Milking_Interval_hrs, Date,
Farm_ID, Feed_Quantity_lb, Mastitis, Milk_Yield_L

NOTES:
- Feed_Quanitity_lb and Feed_Quantity_kg are redundant
- Some others may be worth combining (Temperature + Humidity, Rest + Rumination, Vaccines)
- Date may be condensable into season or month
- Non-numeric features: Breed, Climate_Zone, Management_System, Lactation_Stage, Feed_Type, Date, Farm_ID

In [1]:
import os
if 'experiments' in os.getcwd ():
    os.chdir (os.getcwd () + "/..")
from preprocessing import NUM_FEATURES, CAT_FEATURES, RecordType, engineer_data
import pandas as pd

data = pd.read_csv ("./data/in/cattle_data_test.csv")
data.head()
data.info()
data.describe().T

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40000 entries, 0 to 39999
Data columns (total 35 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Cattle_ID                40000 non-null  int64  
 1   Breed                    40000 non-null  object 
 2   Climate_Zone             40000 non-null  object 
 3   Management_System        40000 non-null  object 
 4   Age_Months               40000 non-null  int64  
 5   Weight_kg                40000 non-null  float64
 6   Parity                   40000 non-null  int64  
 7   Lactation_Stage          40000 non-null  object 
 8   Days_in_Milk             40000 non-null  int64  
 9   Feed_Type                40000 non-null  object 
 10  Feed_Quantity_kg         37985 non-null  float64
 11  Feeding_Frequency        40000 non-null  int64  
 12  Water_Intake_L           40000 non-null  float64
 13  Walking_Distance_km      40000 non-null  float64
 14  Grazing_Duration_hrs  

,count,mean,std,min,25%,50%,75%,max
Cattle_ID,40000.0,20000.500000,11547.149720,1.000000,10000.750000,20000.500000,30000.250000,40000.000000
Age_Months,40000.0,83.263900,34.613224,24.000000,53.000000,83.000000,113.000000,143.000000
Weight_kg,40000.0,501.134588,143.395426,250.000000,377.500000,501.300000,624.500000,750.000000
Parity,40000.0,3.500175,1.704677,1.000000,2.000000,4.000000,5.000000,6.000000
Days_in_Milk,40000.0,183.146600,105.351685,1.000000,92.000000,183.000000,275.000000,364.000000
Feed_Quantity_kg,37985.0,12.015819,3.952974,2.274356,9.316865,12.033033,14.692782,25.469967
Feeding_Frequency,40000.0,3.008675,1.409077,1.000000,2.000000,3.000000,4.000000,5.000000
Water_Intake_L,40000.0,79.892539,14.956080,18.953859,69.822227,79.877772,89.900649,157.259638
Walking_Distance_km,40000.0,4.050308,1.923811,0.500000,2.670000,4.020000,5.360000,12.000000
Grazing_Duration_hrs,40000.0,6.037943,2.867362,1.000000,4.000000,6.000000,8.000000,14.000000


In [2]:
# Count NaNs
print ("Missing Counts:")
for col in data.columns:
    missing_count = data[col].isna ().sum ()
    if missing_count:
        print(f"{col}: {missing_count}")

Missing Counts:
Feed_Quantity_kg: 2015
Housing_Score: 1221
Feed_Quantity_lb: 2015


In [3]:
# Look for duplicates
data[data.duplicated ()]

,Cattle_ID,Breed,Climate_Zone,Management_System,Age_Months,Weight_kg,Parity,Lactation_Stage,Days_in_Milk,Feed_Type,...,IBR_Vaccine,BVD_Vaccine,Rabies_Vaccine,Previous_Week_Avg_Yield,Body_Condition_Score,Milking_Interval_hrs,Date,Farm_ID,Feed_Quantity_lb,Mastitis


### High Correlation Stats

In [4]:
data = engineer_data (data)
keep_num_features = [ft for ft, t in NUM_FEATURES.items () if t == RecordType.KEEP]

for feature in keep_num_features:
    if feature in data.columns:
        print (f"\n{feature}:")
        print ("-" * 40)
        stats = data[feature].describe ()
        print (f"  Mean:    {stats['mean']:.4f}")
        print (f"  Std:     {stats['std']:.4f}")
        print (f"  Range:   {stats['min']:.4f} | {stats['25%']:.4f} | {stats['50%']:.4f} | {stats['75%']:.4f} | {stats['max']:.4f}")
        print (f"  Missing: {data[feature].isna ().sum ()}")


Age_Months:
----------------------------------------
  Mean:    83.2639
  Std:     34.6132
  Range:   24.0000 | 53.0000 | 83.0000 | 113.0000 | 143.0000
  Missing: 0

Weight_kg:
----------------------------------------
  Mean:    501.1346
  Std:     143.3954
  Range:   250.0000 | 377.5000 | 501.3000 | 624.5000 | 750.0000
  Missing: 0

Parity:
----------------------------------------
  Mean:    3.5002
  Std:     1.7047
  Range:   1.0000 | 2.0000 | 4.0000 | 5.0000 | 6.0000
  Missing: 0

Days_in_Milk:
----------------------------------------
  Mean:    183.1466
  Std:     105.3517
  Range:   1.0000 | 92.0000 | 183.0000 | 275.0000 | 364.0000
  Missing: 0

Feed_Quantity_kg:
----------------------------------------
  Mean:    12.0167
  Std:     3.8521
  Range:   2.2744 | 9.4633 | 12.0330 | 14.5191 | 25.4700
  Missing: 0

Water_Intake_L:
----------------------------------------
  Mean:    79.8936
  Std:     14.9459
  Range:   30.0000 | 69.8222 | 79.8778 | 89.9006 | 140.0000
  Missing: 0

Ambi